# Cross-solver comparison and report evidence

Load saved benchmark runs, check recorded input consistency, use shared package
summaries, and export report tables. This notebook does not rerun solvers.

All orders must be complete; failures are reported before quality ranking.
Matching metadata is not independent geometric validation. Explicitly review
objectives, cutting constraints and timing definitions before comparing tools.


In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

import pandas as pd
from cutting_benchmark.summaries import summarise_results
from cutting_benchmark.comparison import validate_comparison_inputs, compare_sheet_counts

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "pyproject.toml").is_file()
     and (p / "src" / "cutting_benchmark").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run from the repository root or a subdirectory")
RESULTS_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR = RESULTS_DIR / "comparison"
INPUTS = {
    "rectpack": RESULTS_DIR / "rectpack" / "rectpack_benchmark_raw.csv",
    "packingsolver": RESULTS_DIR / "packingsolver" / "packingsolver_benchmark_raw.csv",
}
frames = {name: pd.read_csv(path) for name, path in INPUTS.items()}
comparison_results = pd.concat(frames.values(), ignore_index=True)


## 1. Inspect completion and validate inputs

Do not silently drop failures or compare different workloads. Historical missing
validation/status values remain unknown rather than being labelled successful.


In [2]:
completion = (
    comparison_results.groupby(["solver", "config_id", "rotation"], as_index=False)
    .agg(runs=("complete", "size"), complete_runs=("complete", "sum"))
)
display(completion)
validate_comparison_inputs(comparison_results)


,solver,config_id,rotation,runs,complete_runs
0,packingsolver,BinPacking__time30s,False,30,30
1,packingsolver,BinPacking__time30s,True,30,30
2,rectpack,GuillotineBafSas__area__BBF,False,30,30
3,rectpack,GuillotineBafSas__area__BBF,True,30,30
4,rectpack,GuillotineBafSas__input_order__BBF,False,30,30
5,rectpack,GuillotineBafSas__input_order__BBF,True,30,30
6,rectpack,GuillotineBafSas__long_side__BBF,False,30,30
7,rectpack,GuillotineBafSas__long_side__BBF,True,30,30
8,rectpack,GuillotineBafSas__short_side__BBF,False,30,30
9,rectpack,GuillotineBafSas__short_side__BBF,True,30,30


## 2. Shared summary tables

Best-observed counts are calculated across both solvers. The default summary is
sorted by relative excess; the alternative view below is sorted by mean sheets.


In [3]:
tables = summarise_results(comparison_results)
dataset_config = tables["dataset_config"]
tables["summary_by_mean_sheets"] = (
    tables["summary"]
    .sort_values(["rotation", "mean_sheets", "median_runtime"])
    .reset_index(drop=True)
)
display(tables["summary_by_mean_sheets"])


,solver,config_id,rotation,datasets,mean_sheets,total_sheets,median_sheets,best_or_tied_datasets,mean_utilisation,mean_extra_sheets,mean_relative_excess,tied_best_rate,mean_gap_to_area_bound,median_runtime,p95_runtime
0,packingsolver,BinPacking__time30s,False,30,13.466667,404,9.5,30,0.726497,0.000000,0.000000,1.000000,4.700000,1.208018,32.388087
1,rectpack,MaxRectsBssf__long_side__BBF,False,30,13.800000,414,10.0,21,0.686840,0.333333,0.059165,0.700000,5.033333,0.002172,0.007311
2,rectpack,GuillotineBafSas__long_side__BBF,False,30,13.866667,416,10.0,18,0.676762,0.400000,0.076804,0.600000,5.100000,0.001635,0.006596
3,rectpack,MaxRectsBaf__long_side__BBF,False,30,13.866667,416,10.0,19,0.680235,0.400000,0.066957,0.633333,5.100000,0.003114,0.008229
4,rectpack,MaxRectsBssf__area__BBF,False,30,13.900000,417,10.0,19,0.666762,0.433333,0.093164,0.633333,5.133333,0.003109,0.011179
5,rectpack,MaxRectsBaf__area__BBF,False,30,13.900000,417,10.0,19,0.663741,0.433333,0.097937,0.633333,5.133333,0.003505,0.008747
6,rectpack,MaxRectsBssf__short_side__BBF,False,30,13.933333,418,10.0,17,0.664666,0.466667,0.097038,0.566667,5.166667,0.002551,0.007683
7,rectpack,MaxRectsBaf__short_side__BBF,False,30,13.933333,418,10.0,17,0.661818,0.466667,0.098770,0.566667,5.166667,0.002766,0.007822
8,rectpack,SkylineBl__short_side__BBF,False,30,14.333333,430,11.0,11,0.635546,0.866667,0.137719,0.366667,5.566667,0.007131,0.031752
9,rectpack,SkylineBl__long_side__BBF,False,30,14.533333,436,10.5,5,0.626885,1.066667,0.152614,0.166667,5.766667,0.008712,0.030932


Simplified summary

In [10]:
summary_columns = [
    "solver",
    "config_id",
    "rotation",
    "datasets",
    "total_sheets",
    "mean_sheets",
    "best_or_tied_datasets",
    "median_runtime",
]

print(
    "All tested solver configurations.\n"
    "Sorted within each rotation setting by mean sheet count, "
    "then median runtime.\n"
    "Runtime is measured in seconds."
)

display(
    tables["summary_by_mean_sheets"][summary_columns]
)

All tested solver configurations.
Sorted within each rotation setting by mean sheet count, then median runtime.
Runtime is measured in seconds.


,solver,config_id,rotation,datasets,total_sheets,mean_sheets,best_or_tied_datasets,median_runtime
0,packingsolver,BinPacking__time30s,False,30,404,13.466667,30,1.208018
1,rectpack,MaxRectsBssf__long_side__BBF,False,30,414,13.800000,21,0.002172
2,rectpack,GuillotineBafSas__long_side__BBF,False,30,416,13.866667,18,0.001635
3,rectpack,MaxRectsBaf__long_side__BBF,False,30,416,13.866667,19,0.003114
4,rectpack,MaxRectsBssf__area__BBF,False,30,417,13.900000,19,0.003109
5,rectpack,MaxRectsBaf__area__BBF,False,30,417,13.900000,19,0.003505
6,rectpack,MaxRectsBssf__short_side__BBF,False,30,418,13.933333,17,0.002551
7,rectpack,MaxRectsBaf__short_side__BBF,False,30,418,13.933333,17,0.002766
8,rectpack,SkylineBl__short_side__BBF,False,30,430,14.333333,11,0.007131
9,rectpack,SkylineBl__long_side__BBF,False,30,436,14.533333,5,0.008712


## 3. Explicit fixed-baseline comparison

The baseline below was identified on this benchmark, not on held-out data.
Candidate wins mean fewer sheets, with no runtime tie-break. Reduction values
are fractions (multiply by 100 for percentages).


In [7]:
BASELINE_CONFIG = "MaxRectsBssf__long_side__BBF"
CANDIDATE_CONFIG = "BinPacking__time30s"
baseline = dataset_config.loc[
    dataset_config["solver"].eq("rectpack")
    & dataset_config["config_id"].eq(BASELINE_CONFIG)
]
candidate = dataset_config.loc[
    dataset_config["solver"].eq("packingsolver")
    & dataset_config["config_id"].eq(CANDIDATE_CONFIG)
]
fixed = compare_sheet_counts(baseline, candidate)

fixed_labels = {
    "baseline_solver": "rectpack",
    "baseline_config_id": BASELINE_CONFIG,
    "baseline_selection": "fixed_configuration",
    "candidate_solver": "packingsolver",
    "candidate_config_id": CANDIDATE_CONFIG,
}

for name, frame in fixed.items():
    labelled = frame.assign(**fixed_labels)

    label_columns = list(fixed_labels)
    other_columns = [
        column
        for column in labelled.columns
        if column not in label_columns
    ]

    tables[f"fixed_baseline_{name}"] = labelled[
        label_columns + other_columns
    ]

print(
    "Fixed configuration comparison:\n"
    f"Baseline: rectpack / {BASELINE_CONFIG}\n"
    f"Candidate: packingsolver / {CANDIDATE_CONFIG}\n"
    "Positive sheets_saved means the candidate uses fewer sheets."
)

display(tables["fixed_baseline_summary"])


Fixed configuration comparison:
Baseline: rectpack / MaxRectsBssf__long_side__BBF
Candidate: packingsolver / BinPacking__time30s
Positive sheets_saved means the candidate uses fewer sheets.


,baseline_solver,baseline_config_id,baseline_selection,candidate_solver,candidate_config_id,rotation,datasets,baseline_total_sheets,candidate_total_sheets,sheets_saved,candidate_wins,ties,candidate_losses,relative_reduction
0,rectpack,MaxRectsBssf__long_side__BBF,fixed_configuration,packingsolver,BinPacking__time30s,False,30,414,404,10,9,21,0,0.024155
1,rectpack,MaxRectsBssf__long_side__BBF,fixed_configuration,packingsolver,BinPacking__time30s,True,30,398,391,7,7,23,0,0.017588


## 4. Hindsight-best Rectpack comparison

The per-dataset minimum is a retrospective reference, not an implemented
portfolio algorithm. It has no deployable strategy runtime.


In [9]:
rectpack_best = (
    dataset_config.loc[dataset_config["solver"].eq("rectpack")]
    .groupby(["dataset_id", "rotation"], as_index=False)
    .agg(sheets_used=("sheets_used", "min"))
)
hindsight = compare_sheet_counts(rectpack_best, candidate)

hindsight_labels = {
    "baseline_solver": "rectpack",
    "baseline_config_id": "minimum_across_tested_configurations",
    "baseline_selection": "hindsight_per_dataset",
    "candidate_solver": "packingsolver",
    "candidate_config_id": CANDIDATE_CONFIG,
}

for name, frame in hindsight.items():
    labelled = frame.assign(**hindsight_labels)

    label_columns = list(hindsight_labels)
    other_columns = [
        column
        for column in labelled.columns
        if column not in label_columns
    ]

    tables[f"hindsight_rectpack_{name}"] = labelled[
        label_columns + other_columns
    ]

print(
    "Hindsight comparison:\n"
    "Baseline: lowest sheet count across tested Rectpack configurations "
    "for each dataset and rotation setting.\n"
    f"Candidate: packingsolver / {CANDIDATE_CONFIG}\n"
    "This baseline is a retrospective reference, not a measured "
    "portfolio algorithm."
)

display(tables["hindsight_rectpack_summary"])


Hindsight comparison:
Baseline: lowest sheet count across tested Rectpack configurations for each dataset and rotation setting.
Candidate: packingsolver / BinPacking__time30s
This baseline is a retrospective reference, not a measured portfolio algorithm.


,baseline_solver,baseline_config_id,baseline_selection,candidate_solver,candidate_config_id,rotation,datasets,baseline_total_sheets,candidate_total_sheets,sheets_saved,candidate_wins,ties,candidate_losses,relative_reduction
0,rectpack,minimum_across_tested_configurations,hindsight_per_dataset,packingsolver,BinPacking__time30s,False,30,410,404,6,6,24,0,0.014634
1,rectpack,minimum_across_tested_configurations,hindsight_per_dataset,packingsolver,BinPacking__time30s,True,30,395,391,4,4,26,0,0.010127


## 5. Export numerical evidence and provenance

This cell overwrites comparison outputs. CSVs retain numerical precision.
The manifest records analysis-time packages, not necessarily the historical
solver environment. Commit the notebook, helper code and intended outputs
together after review; previous result directories are not cleaned automatically.


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = []
for name, table in tables.items():
    path = OUTPUT_DIR / f"comparison_{name}.csv"
    table.to_csv(path, index=False)
    outputs.append(str(path.relative_to(PROJECT_ROOT)))
completion.to_csv(OUTPUT_DIR / "comparison_raw_completion.csv", index=False)
outputs.append(str((OUTPUT_DIR / "comparison_raw_completion.csv").relative_to(PROJECT_ROOT)))

def git_output(*args):
    try:
        return subprocess.check_output(
            ["git", "-C", str(PROJECT_ROOT), *args], text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None

packages = {}
for package in ["pandas", "rectpack", "packingsolver"]:
    try:
        packages[package] = version(package)
    except PackageNotFoundError:
        packages[package] = None

manifest = {
    "analysis_python": sys.version,
    "analysis_packages": packages,
    "git_commit": git_output("rev-parse", "HEAD"),
    "git_status": git_output("status", "--porcelain"),
    "fixed_baseline": {"solver": "rectpack", "config_id": BASELINE_CONFIG},
    "candidate": {"solver": "packingsolver", "config_id": CANDIDATE_CONFIG},
    "inputs": {
        name: {
            "path": str(path.relative_to(PROJECT_ROOT)),
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "rows": len(frames[name]),
        }
        for name, path in INPUTS.items()
    },
    "outputs": outputs,
}
(OUTPUT_DIR / "comparison_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print(f"Saved {len(outputs)} CSV tables and a manifest to {OUTPUT_DIR}")


Saved 10 CSV tables and a manifest to /workspaces/2D-Optimisation-in-Metals-Manufacturing/results/comparison


## How to read the main outputs

| Output                                    | Question it answers                                                                                    |
| ----------------------------------------- | ------------------------------------------------------------------------------------------------------ |
| comparison_summary_by_mean_sheets.csv     | How does every fixed solver configuration perform overall?                                             |
| comparison_fixed_baseline_summary.csv     | How does PackingSolver compare with the selected fixed Rectpack baseline?                              |
| comparison_fixed_baseline_paired.csv      | On which individual datasets does that difference occur?                                               |
| comparison_hindsight_rectpack_summary.csv | Does PackingSolver improve even on the best Rectpack sheet count selected separately for each dataset? |
| comparison_summary_by_workload.csv        | How do results differ by synthetic profile and order size?                                             |